<fieldset style="padding:10px; border:1px solid #ccc; box-shadow:2px 2px 5px rgba(0,0,0,0.1);">
<legend style="font-size: 10px; color:#555;">Credits</legend>

<table style="width: 100%; border-collapse: collapse;">
    <tr>
        <td style="width: 80px; vertical-align: top;">
            <img src="https://raw.githubusercontent.com/AstroStat-Academy/assets-public/main/logo/logo_b_text_lowres.png" alt="AstroStat Academy logo" width="100">
        </td>
        <td style="vertical-align: center; padding-left: 15px; font-size: 10px; line-height: 1.2;">
            This notebook contains original work by the authors unless stated otherwise.
            Any external material is properly credited to its sources.<br>
            References to papers, datasets, and software are acknowledged.
            Original content is licensed under the <a href="https://www.gnu.org/licenses/gpl-3.0.en.html">GNU General Public License v3.0 (GNU GPLv3)</a>.
        </td>
    </tr>
</table>

</fieldset>
<!-- Allow these <br> or it will look ugly once rendered on Jupyter Book. -->
<br>

<font size=6>**Methods**</font>

<div style="border-left: 5px solid #2878b5; padding: 12px 20px; margin: 20px 0;">
  <p style="font-size: 18px; margin: 0 0 8px;">
    <strong>TALES School II</strong><br>
    Hands-on workshop on machine learning applications to astrophysics
  </p>
  <p style="color: #666; margin: 0;">
    September 13–18, 2026 · Petnica Science Center, Serbia
  </p>
</div>

<div style="border: 2px solid #ddd; border-radius: 8px; padding: 15px; background-color: #f9f9f9;">

Some suggestions on thetechniques you may use to estimate time delays.

- **Cross-correlation**
- **Bayesian Blocks**
- **PyCS: PyCS : Python Curve Shifting**

</div>

# Cross-Correlation

<div style="border-left: 4px solid #4CAF50; padding-left: 1em; padding-top: 1em; padding-bottom: 1em; margin: 1em 0; background: #e8f5e9; color: #1a1a1a;">

**Cross-Correlation** (**CC**) measures how closely 2 time series vary together:

$$
r = \frac{\sum_i (X_i-\bar X)(Y_i-\bar Y)}
{\sqrt{\sum_i (X_i-\bar X)^2}\sqrt{\sum_i (Y_i-\bar Y)^2}}.
$$

A value near $+1$ indicates the perfect correlation.

</div>

## Using Cross-Corrleation to detect time delays

<div style="border-left: 4px solid #2196F3; padding-left: 1em; padding-top: 1em; padding-bottom: 1em; margin: 1em 0; background: #f0f8ff; color: #1a1a1a;">

**Intuition:** If 2 times series are shifted by $\Delta t$, then we can **correct** for that time lag and obtain optimal $r$.

</div>

**In practice:** We do it the other way around — We artificially shift a series, recalculating the $r$, looking for the shift that **maximises** $r$.

$$
\begin{array}{ll}
\hline
&{\textbf{Algorithm: Scan the lag}} \\
\hline
1. & \text{For } \tau=-T,\,-T+1,\,\ldots,\,T:\\
2. & \quad \text{Shift } Y \text{ by } \tau \text{ time units.}\\
3. & \quad r(\tau) \leftarrow \operatorname{corr}\!\bigl(X(t),\,Y(t+\tau)\bigr).\\
4. & \widehat{\Delta t} \leftarrow \underset{\tau \in [-T,T]}{\operatorname{arg\,max}}\;r(\tau).\\
\hline
\end{array}
$$

In [10]:
from IPython.display import Video, display

display(Video("videos/cross_correlation.mp4", embed=False, width=1000))


### What to watch for

- The **CC** technique assumes the data are just shifted in $\Delta t$, and at most they have different means, but are **identical otherwise**.
> _If the series, e.g., have different trends, those have to be removed <u>before</u> running CC._

- **Gaps/Iregular sampling**
> _See chapter "[$\S$Cross-Correlation when gaps occur](Cross-Correlation-when-gaps-occur)" for a solution._

- **Very noisy series** may create spurious peaks.
> _See chapter "[$\S$Bayesian Blocks + Cross-Correlation](#Bayesian-Blocks-+-Cross-Correlation)" for a solution._


## Cross-Correlation when gaps occur

<div style="border-left: 4px solid #2196F3; padding-left: 1em; padding-top: 1em; padding-bottom: 1em; margin: 1em 0; background: #f0f8ff; color: #1a1a1a;">

**Intuition:** As we are iterating over different lags $\tau$, drop the ($X_i$, $Y_i$) pairs where either $X_i$ or $Y_i$ is missing.

</div>

$$
\begin{array}{ll}
\hline
&{\textbf{Algorithm: Find candidate lags with gaps}} \\
\hline
1. & \text{For each trial lag } \tau = -T,\,-T+1,\,\ldots,\,T:\\
2. & \quad \text{Pair } X(t) \text{ with } Y(t+\tau).\\
3. & \quad \text{Remove pairs with missing data.}\\
4. & \quad \text{If enough pairs remain and both series vary:}\\
5. & \qquad r(\tau) \leftarrow \operatorname{correlation}(\text{remaining pairs}).\\
6. & \text{Plot } r(\tau) \text{ against } \tau.\\
7. & \text{Return the lag at the highest valid correlation peak as a candidate.}\\
\hline
\end{array}
$$

## Libraries

- `NumPy`: [`numpy.correlate`](https://numpy.org/doc/stable/reference/generated/numpy.correlate.html) for basic cross-correlation of two sequences.
- `SciPy`: [`scipy.signal.correlate`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.correlate.html) for direct or FFT-based cross-correlation.
- `statsmodels`: [`statsmodels.tsa.stattools.ccf`](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.ccf.html) for normalized time-series cross-correlation.

These functions use sample-index lags; interpreting them as time delays requires a common, evenly spaced time grid.



# Bayesian Blocks + Cross-Correlation

For _noisy_, _unevenly_ sampled time series, chance alignments of fluctuations can produce **misleading** CC maxima.

**Smoothing** can reduce noise, but choosing a smoothing **scale** involves a **trade-off**:
- too little retains noise
- too much erases real variability.

**Bayesian Blocks** ([Scargle et al. 2013](https://arxiv.org/abs/1207.5578)) provides a smoothing whose scale is:
- **informed** by the data;
- **adaptable** over time.

<div style="border-left: 4px solid #2196F3; padding-left: 1em; padding-top: 1em; padding-bottom: 1em; margin: 1em 0; background: #f0f8ff; color: #1a1a1a;">

**Intuition:** Bayesian Blocks models the series as constant-level segments, using:
- **longer** segments where the signal is **steady**
- **shorter** ones where it **changes rapidly**.

Of course, fitting this model is a matter of chosing the segment **lengths**.

</div>

## The objective

The series is modelled as a **piecewise constant** model — We define: 
- **blocks** = the segments
- **change point** = the right edge of a block

There are many combinations of blocks we could draw: let us call "**partition**" a specific segmentation into blocks.<br>
_Namely, for $N$ points, there are $2^N$ possible partitions._


So the iddur becomes:
> _How do we pick the best partition?_

$\rightarrow$ We need a **score**:

<div style="border-left: 4px solid #4CAF50; padding-left: 1em; padding-top: 1em; padding-bottom: 1em; margin: 1em 0; background: #e8f5e9; color: #1a1a1a;">

$$\text{score}(\text{partition}) \;=\; \sum_{\text{blocks}}\Big[\;
   \underbrace{\text{fitness(block)}}_{\text{how well one constant fits the points}}
   \;-\; \underbrace{\texttt{ncp\_prior}}_{\text{cost of adding a block}}\;\Big]$$

- **fitness** is the likelihood of the best constant for that block's points — For Gaussian errors:
$$ {\hat {y}}={\frac {\sum _{i}y_{i}/\sigma _{i}^{2}}{\sum _{i}1/\sigma _{i}^{2}}}.$$

- **`ncp_prior`** is a fixed cost paid once per block.
> _Without it, the optimum would be $N$ blocks, one around each point, which is clearly overfitting_.

</div>

<div style="border-left: 4px solid #2196F3; padding-left: 1em; padding-top: 1em; padding-bottom: 1em; margin: 1em 0; background: #f0f8ff; color: #1a1a1a;">

`ncp_prior` is set from a **false-alarm probability `p0`** — the probability of adding one spurious block — through an empirical calibration.<br>

> _Smaller `p0` → larger penalty → fewer blocks._
</div>

## How Bayesian Blocks is solved

Let `A(n)` be the best achievable score for the first _n_ points — Then:

$$A(n) \;=\; \max_{1\le r\le n}\Big[\,A(r-1)\;+\;\text{fitness}(r\ldots n)\;-\;\texttt{ncp\_prior}\,\Big]$$


So, **in practice**:

1. Sweep *n* from 1 to *N*, remember the arg-max *r* at each step,
2. Backtrack from *N* to read off the change points.


In [20]:
from IPython.display import Video, display
display(Video("videos/bayesian_blocks.mp4", embed=False, width=1000))

Because each `A(n)` is built on an already-optimal `A(r-1)`, the result is the global optimum, not a greedy guess.

<div style="border-left: 4px solid #2196F3; padding-left: 1em; padding-top: 1em; padding-bottom: 1em; margin: 1em 0; background: #f0f8ff; color: #1a1a1a;">

FUN FACT: There are $2^{N}$ partitions of *N* points, but the best one is found **exactly** in $O(N^2)$.

</div>

## The penalty is the only knob

`ncp_prior` (equivalently `p0`) sets the resolution:

- Too small a penalty splits noise into spurious blocks.
- Too large merges real features.

It is **not** fitted per dataset — a fixed `p0 ≈ 0.01–0.05` is standard and the result is stable across a wide range.

## Libraries

- `Astropy`: [`astropy.stats.bayesian_blocks`](https://docs.astropy.org/en/stable/api/astropy.stats.bayesian_blocks.html) for time-series segmentation; use `fitness="measures"` for measurements with Gaussian errors, as in this notebook.
- `hepstats`: [`hepstats.modeling.bayesian_blocks`](https://scikit-hep.org/hepstats/) for adaptive histogram binning of event data.


# PyCS: Python Curve Shifting

PyCS is a specialized library for measuring time delays between gravitationally lensed images of the same quasar.

<div style="border-left: 4px solid #2196F3; padding: 1em; margin: 1em 0; background: #f0f8ff; color: #1a1a1a;">

**Intuition:** Find the **common generative process** behind the observed light curves: the quasar’s intrinsic variability, seen with different delays.

</div>

![Two simulated images of one quasar, before and after correcting a 20-day delay and a magnitude offset. The aligned observations follow one common brightness history.](images/pycs_alignment.png)

PyCS uses **splines** or **Gaussian-processes** as **regressors** (to fit the curve).

It exploits 2 approaches to determine the $\Delta t$:
- Fit **1 spline** while shifting the light curves along the time axis to find their best alignment.
- Fit **each light** curve separately with a **Gaussian process**, then shift them to minimise variations in their differences.


## The objective (spline)

We will look at the **spline fitting**.

<div style="border-left: 4px solid #2196F3; padding: 1em; margin: 1em 0; background: #f0f8ff; color: #1a1a1a;">

**NOTE:** As a bonus, the spline method fits a shared quasar signal plus a <u>separate</u> microlensing component for each image.

</div>

<div style="border-left: 4px solid #4CAF50; padding: 1em; margin: 1em 0; background: #e8f5e9; color: #1a1a1a;">

Given the model:

$$m_j(t) \approx \underbrace{s(t-\tau_j)}_{\text{common history, delayed}}
+ \underbrace{c_j}_{\text{magnitude offset}}
+ \underbrace{\mu_j(t)}_{\text{microlensing}}.$$

Fit the delays $\tau_j$ **and** model curves by minimising squared residuals weighted by measurement uncertainties:

$$\chi^2 = \sum_{j,i}\left[\frac{m_{ij}-s(t_{ij}-\tau_j)-c_j-\mu_j(t_{ij})}{\sigma_{ij}}\right]^2.$$

</div>

The fitted parameters are:
- Time delays $\tau_j$.
- Magnitude offsets $c_j$.
- Common spline **coefficients** and **knot positions**.
- Microlensing model $\mu$ parameters.


## In practice

Start from a **guess** delay $\tau$ (_e.g., from Cross-Correlation analysis_), then:

1. Hold $\tau$ fixed: fit the spline parameters.
2. Hold the spline fixed: fit $\tau$ by minimising the mismatch as the observations shift in time.
3. Repeat, because changing either changes the best value of the other.


<video src="videos/pycs_spline.mp4" controls preload="metadata" width="1000"></video>


## What to watch for

- **Flexibility:** Too many knots fit noise; overly flexible microlensing can absorb intrinsic features.
- **Ambiguity:** Gaps or weak variability can permit several delays.

## Libraries

- [PyCS3](https://gitlab.com/cosmograil/PyCS3): [spline fitting](https://cosmograil.gitlab.io/PyCS3/tutorial/shifting.html) and [mock-curve analysis](https://cosmograil.gitlab.io/PyCS3/tutorial/drawing.html) for measuring time delays.
- `scikit-learn`: Gaussian-process backend for PyCS3's alternative [regression-difference method](https://cosmograil.gitlab.io/PyCS3/tutorial/shifting.html#the-regdiff-optimizer), which aligns separately smoothed curves.
- [TDCOSMO XVII time delays](https://github.com/duxfrederic/TDCOSMO_XVII_time_delays): based on PyCS3, but easier to use.
- [pyPETAL](https://pypetal.readthedocs.io/en/latest/index.html).


# Beyond the Statistics

> _The ones presented above are **statistics**-heavy methods._

Feel free to experiment with **Machine Learning** and **Neural Networks**, if  you so wish!

You may even try multiple methods and then create an **Ensamble Model** that aggregates the results.

In [1]:
#EOF